In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from scipy.spatial import ConvexHull
from collections import defaultdict, Counter

from mabc.src.dst_solver import DSTValueIteration
from scipy.optimize import linprog

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Computer Modern Roman', 'serif'],
    'mathtext.fontset': 'cm',
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 18,
    'figure.titlesize': 22,
    'legend.fontsize': 14,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})
GAMMA = 0.999


# Core RL & Imitation Learning Utilities


In [ ]:
def collect_trajectories(solver, policy, episodes):
    """Rolls out the policy to collect state-action pairs from RANDOM spawns."""
    dataset = []
    for _ in range(episodes):
        obs, _ = solver.reset_random()
        term = trunc = False
        while not (term or trunc):
            state = tuple(obs)
            if state not in policy: break
            action = policy[state]
            dataset.append((state, action))
            obs, _, term, trunc, _ = solver.env.step(action)
    return dataset

def build_stochastic_policy(solver, dataset):
    """Builds an MLE stochastic policy (Uniform random for unseen states)."""
    counts = defaultdict(Counter)
    for s, a in dataset:
        counts[s][a] += 1
        
    policy = {}
    for r in range(solver.rows):
        for c in range(solver.cols):
            state = (r, c)
            val = solver.env.unwrapped.sea_map[r, c]
            if val == -10 or val > 0: continue # Skip terminal/rock states
            
            if state in counts:
                total = sum(counts[state].values())
                policy[state] = {a: count / total for a, count in counts[state].items()}
            else:
                policy[state] = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}
    return policy

def exact_policy_evaluation(solver, policy_probs):
    """Exact Analytic Policy Evaluation (No Monte Carlo Noise)."""
    R_pi = np.zeros((solver.n_states, 2))
    P_pi = np.zeros((solver.n_states, solver.n_states))
    
    for (r, c) in solver.non_terminal_states:
        state = (r, c)
        i = solver.state_to_idx[state] 
        probs = policy_probs.get(state, {0:0.25, 1:0.25, 2:0.25, 3:0.25})
        
        for a, prob in probs.items():
            if prob == 0: continue
            R_pi[i] += prob * solver.rewards[r, c, a]
            if not solver.terminals[r, c, a]:
                nr, nc = solver.transitions[r, c, a]
                j = solver.state_to_idx[(nr, nc)]
                P_pi[i, j] += prob * 1.0
                
    for state, i in solver.state_to_idx.items():
        if state not in solver.non_terminal_states:
            P_pi[i, i] = 1.0
                
    A = np.eye(solver.n_states) - solver.gamma * P_pi
    V = np.linalg.solve(A, R_pi)
    ret = np.dot(solver.rho, V)
    return np.array([float(ret[0]), float(ret[1])])

def get_path(solver, policy):
    """Extracts the deterministic path taken by a policy from the default start."""
    obs, _ = solver.env.reset()
    path = [tuple(obs)]
    term = trunc = False
    while not (term or trunc):
        state = tuple(obs)
        if state not in policy: break
        action = policy[state]
        obs, _, term, trunc, _ = solver.env.step(action)
        path.append(tuple(obs))
    return path

# Visualization Utilities


In [ ]:

def draw_policy_on_grid(ax, solver, policy, title="", highlight_conflicts=False, is_mixed_eval=False):
    """A master function to draw the Deep Sea Treasure grid and policy arrows."""
    rows, cols = solver.rows, solver.cols
    sea_map = solver.env.unwrapped.sea_map
    action_map = {0: (0, 0.35), 1: (0, -0.35), 2: (-0.35, 0), 3: (0.35, 0)}
    
    ax.set_facecolor('#e8f4f8')
    for r in range(rows):
        for c in range(cols):
            val = sea_map[r, c]
            state = (r, c)
            y_pos = rows - 1 - r
            is_terminal = (val == -10 or val > 0)
            
            # Base Coloring
            color = "#808080" if val == -10 else ("#ffd700" if val > 0 else "white")
            
            # State Analysis
            is_random = False
            is_conflict = False
            if not is_terminal and state in policy:
                if len(policy[state]) == 4 and all(p == 0.25 for p in policy[state].values()):
                    is_random = True
                    color = "#ffcccc" # OOD / Random Guessing
                elif len(policy[state]) > 1:
                    is_conflict = True
                    if highlight_conflicts:
                        color = "#ffe6e6" if is_mixed_eval else "#e6ffe6" 
            
            rect = patches.Rectangle((c - 0.5, y_pos - 0.5), 1, 1, 
                                     linewidth=1, edgecolor='black', facecolor=color, alpha=0.8)
            ax.add_patch(rect)
            
            if val > 0:
                ax.text(c, y_pos, str(int(val)), ha='center', va='center', fontweight='bold', zorder=10)
            
            # Draw Arrows
            if not is_terminal and state in policy:
                for action, prob in policy[state].items():
                    if prob == 0: continue
                    dx, dy = action_map[action]
                    aw = 0.02 + (0.08 * prob)
                    hw = 0.15 + (0.15 * prob)
                    alpha = 0.3 + (0.7 * prob)
                    
                    if is_random: arr_color = 'red'
                    elif is_conflict: arr_color = 'blue' if is_mixed_eval else 'green'
                    else: arr_color = 'black'
                        
                    ax.arrow(c - (dx*0.1), y_pos - (dy*0.1), dx*0.8, dy*0.8, 
                             width=aw, head_width=hw, head_length=hw, 
                             fc=arr_color, ec=arr_color, alpha=alpha, length_includes_head=True)

    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(-0.5, rows - 0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    if title: ax.set_title(title, fontsize=14, fontweight='bold')

# Pareto Geometry Utilities


In [ ]:
def build_continuous_frontier(pareto_points):
    """Interpolates the dense continuous boundary between deterministic vertices."""
    ideal_point = np.max(pareto_points, axis=0)
    nadir_point = np.min(pareto_points, axis=0)
    obj_ranges = np.clip(ideal_point - nadir_point, 1e-9, None)
    
    norm_vertices = (pareto_points - nadir_point) / obj_ranges
    dense_norm_front = []
    metadata = []
    
    for i in range(len(norm_vertices) - 1):
        p1, p2 = norm_vertices[i], norm_vertices[i+1]
        alphas = np.linspace(0, 1, 1000)
        for alpha in alphas:
            dense_norm_front.append((1 - alpha) * p1 + alpha * p2)
            metadata.append((i, i+1, alpha))
            
    return np.array(dense_norm_front), metadata, nadir_point, obj_ranges

def calc_normalized_l_inf_distance(ret, dense_norm_front, nadir_point, obj_ranges):
    """Calculates the L_inf distance from a return to the continuous frontier."""
    norm_ret = (ret - nadir_point) / obj_ranges
    distances = np.max(np.abs(dense_norm_front - norm_ret), axis=1)
    return np.min(distances)

def calc_exact_l_inf_distance(ret, pareto_points, obj_ranges):
    """
    Exact Chebyshev L-infinity projection using Linear Programming.
    MAXIMIZES the normalized gap delta such that some point on the convex hull 
    strictly dominates 'ret' by at least delta in all dimensions.
    """
    N_vertices = len(pareto_points)
    n_obj = len(ret) # Will dynamically handle 2D (DST) or 3D
    
    c = np.zeros(N_vertices + 1)
    c[0] = -1.0  
    
    A_eq = np.zeros((1, N_vertices + 1))
    A_eq[0, 1:] = 1.0
    b_eq = np.array([1.0])
    
    A_ub = np.zeros((n_obj, N_vertices + 1))
    b_ub = np.zeros(n_obj)
    
    for i in range(n_obj):
        A_ub[i, 0] = obj_ranges[i]
        A_ub[i, 1:] = -pareto_points[:, i]
        b_ub[i] = -ret[i]
        
    bounds = [(None, None)] + [(0, 1) for _ in range(N_vertices)]
    
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    
    return max(0.0, -res.fun) if res.success else 0.0

# Environment Initialization & Auditing

In [ ]:
full_solver = DSTValueIteration(gamma=GAMMA)

ccs_results = full_solver.find_ccs_ols()
sorted_pareto_points = np.array([r for r, p in ccs_results])
sorted_pareto_policies = [p for r, p in ccs_results]

dense_front, front_meta, nadir, ranges = build_continuous_frontier(sorted_pareto_points)

expert_1_idx, expert_2_idx = 0, 14
pol_A_true = sorted_pareto_policies[expert_1_idx]
pol_B_true = sorted_pareto_policies[expert_2_idx]
ret_A = sorted_pareto_points[expert_1_idx]
ret_B = sorted_pareto_points[expert_2_idx]

# Data Collection & Sharing Logic

In [ ]:

def generate_augmented_datasets(solver, pol_A, pol_B, num_episodes):
    """Collects data and strictly partitions the shared knowledge."""
    data_A = collect_trajectories(solver, pol_A, num_episodes)
    data_B = collect_trajectories(solver, pol_B, num_episodes)
    
    visited_A = dict(data_A)
    visited_B = dict(data_B)
    
    shared_from_B = [(s, a) for (s, a) in data_B if s not in visited_A or visited_A[s] == a]
    shared_from_A = [(s, a) for (s, a) in data_A if s not in visited_B or visited_B[s] == a]
    
    return data_A, data_B, shared_from_A, shared_from_B

data_A, data_B, shared_A, shared_B = generate_augmented_datasets(full_solver, pol_A_true, pol_B_true, 200)

print(f"Collected {len(data_A)} total samples for Policy A.")
print(f"Augmented with {len(shared_B)} shared state-actions from Policy B.")

 # Behavioral Cloning Comparison (Visual)

In [ ]:
pi_mixed = build_stochastic_policy(full_solver, data_A + data_B)
pi_iso_A = build_stochastic_policy(full_solver, data_A)
pi_aug_A = build_stochastic_policy(full_solver, data_A + shared_B)

ret_mixed = exact_policy_evaluation(full_solver, pi_mixed)
ret_iso_A = exact_policy_evaluation(full_solver, pi_iso_A)
ret_aug_A = exact_policy_evaluation(full_solver, pi_aug_A)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

draw_policy_on_grid(axes[0], full_solver, pi_mixed, 
                    f"Mixed BC\nReturn: {np.round(ret_mixed, 1)}", True, True)

draw_policy_on_grid(axes[1], full_solver, pi_iso_A, 
                    f"Isolated BC (Expert A)\nReturn: {np.round(ret_iso_A, 1)}")

draw_policy_on_grid(axes[2], full_solver, pi_aug_A, 
                    f"Augmented BC (MA-BC)\nReturn: {np.round(ret_aug_A, 1)}", True, False)

plt.tight_layout()
plt.show()

# Normalized L-Infinity Suboptimality Curves

In [ ]:

def run_suboptimality_scaling_experiment(solver, pol_A, pol_B, pareto_points, 
                                         traj_steps, num_trials=10):
    ideal_point = np.max(pareto_points, axis=0)
    nadir_point = np.min(pareto_points, axis=0)
    ranges = np.clip(ideal_point - nadir_point, 1e-9, None)
    
    results = {"mixed": [], "iso": [], "aug": []}
    
    for n_traj in traj_steps:
        gaps = {"mixed": [], "iso": [], "aug": []}
        
        for _ in range(num_trials):
            dA, dB, sA, sB = generate_augmented_datasets(solver, pol_A, pol_B, n_traj)
            
            pi_mix = build_stochastic_policy(solver, dA + dB) 
            pi_iso_A, pi_iso_B = build_stochastic_policy(solver, dA), build_stochastic_policy(solver, dB)
            pi_aug_A, pi_aug_B = build_stochastic_policy(solver, dA + sB), build_stochastic_policy(solver, dB + sA)
            
            ret_mix = exact_policy_evaluation(solver, pi_mix)
            ret_iso_A = exact_policy_evaluation(solver, pi_iso_A)
            ret_iso_B = exact_policy_evaluation(solver, pi_iso_B)
            ret_aug_A = exact_policy_evaluation(solver, pi_aug_A)
            ret_aug_B = exact_policy_evaluation(solver, pi_aug_B)
            
            gap_mix = calc_exact_l_inf_distance(ret_mix, pareto_points, ranges)
            
            gap_iso_A = calc_exact_l_inf_distance(ret_iso_A, pareto_points, ranges)
            gap_iso_B = calc_exact_l_inf_distance(ret_iso_B, pareto_points, ranges)
            gap_iso = (gap_iso_A + gap_iso_B) / 2.0
            
            gap_aug_A = calc_exact_l_inf_distance(ret_aug_A, pareto_points, ranges)
            gap_aug_B = calc_exact_l_inf_distance(ret_aug_B, pareto_points, ranges)
            gap_aug = (gap_aug_A + gap_aug_B) / 2.0
            
            gaps["mixed"].append(gap_mix)
            gaps["iso"].append(gap_iso)
            gaps["aug"].append(gap_aug)
            
        for k in results:
            results[k].append((np.mean(gaps[k]), np.std(gaps[k])))
            
        print(f"Traj: {n_traj:3d} | Mixed: {results['mixed'][-1][0]:.1%} | Iso: {results['iso'][-1][0]:.1%} | Aug: {results['aug'][-1][0]:.1%}")
    
    return results

trajectory_steps = [0,10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
res = run_suboptimality_scaling_experiment(full_solver, pol_A_true, pol_B_true, sorted_pareto_points, trajectory_steps, num_trials=100)

fig = plt.figure(figsize=(5, 5.2))
colors = {"mixed": "red", "iso": "orange", "aug": "green"}
labels = {
    "mixed": "Naive BC (Joint Dataset)", 
    "iso": "Isolated BC (Independent Experts)", 
    "aug": "MA-BC (Ours)"
}
markers = {"mixed": "d", "iso": "o", "aug": "s"}


for k in ["mixed", "iso", "aug"]:
    means = np.array([m for m, s in res[k]])
    stds = np.array([s for m, s in res[k]])
    stds = stds / np.sqrt(100) 
    plt.plot(trajectory_steps, means, marker=markers[k], color=colors[k], linewidth=2.5, label=labels[k])
    
    plt.fill_between(trajectory_steps, np.maximum(0, means - stds), means + stds, color=colors[k], alpha=0.15, edgecolor='none')

fig.subplots_adjust(top=0.90, bottom=0.25, left=0.24, right=0.98)
plt.xticks(trajectory_steps)
plt.xlabel(r"Dataset Size per Expert")
plt.ylabel(r"Normalized $L_\infty$ Distance to PF")
plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
plt.grid(True, linestyle=":", alpha=0.6)
plt.ylim(bottom=0)
plt.tight_layout()

plt.savefig("dst_learning_curve.pdf", transparent=True, dpi=300)
plt.show()

# Varying Initial Distribution (Stochasticity Sweep)


In [ ]:
def plot_shared_data_value_across_distributions(alphas=[0.0, 0.5, 1.0],
                                                trajectory_steps=[10, 20, 30, 40, 50, 75, 100], 
                                                num_trials=20):
    print("="*65)
    print(" EVALUATING THE VALUE OF SHARED DATA ACROSS DISTRIBUTIONS ")
    print("="*65 + "\n")
    
    n_panels = len(alphas)
    
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5.2), sharey=True)
    if n_panels == 1: axes = [axes]
    
    print(f"Running evaluation over {n_panels} isolated distribution states...\n")
    
    for ax_idx, alpha in enumerate(alphas):
        print(f"--- Processing Alpha = {alpha} ---")
        
        solver = DSTValueIteration(gamma=GAMMA, init_state_dist=alpha)
        
        ccs_results = solver.find_ccs_ols()
        pareto_points = np.array([r for r, p in ccs_results])
        pareto_policies = [p for r, p in ccs_results]
        
        ideal_point = np.max(pareto_points, axis=0)
        nadir_point = np.min(pareto_points, axis=0)
        ranges = np.clip(ideal_point - nadir_point, 1e-9, None)
        
        idx_A = 0
        idx_B = len(pareto_points) - 1
        pol_A_true = pareto_policies[idx_A]
        pol_B_true = pareto_policies[idx_B]

        results = {"iso": [], "aug": []}
        
        for n_traj in trajectory_steps:
            gaps = {"iso": [], "aug": []}
            
            for _ in range(num_trials):
                dA, dB, sA, sB = generate_augmented_datasets(solver, pol_A_true, pol_B_true, n_traj)
                
                pi_iso_A = build_stochastic_policy(solver, dA)
                pi_iso_B = build_stochastic_policy(solver, dB)
                pi_aug_A = build_stochastic_policy(solver, dA + sB)
                pi_aug_B = build_stochastic_policy(solver, dB + sA)
                
                ret_iso_A = exact_policy_evaluation(solver, pi_iso_A)
                ret_iso_B = exact_policy_evaluation(solver, pi_iso_B)
                ret_aug_A = exact_policy_evaluation(solver, pi_aug_A)
                ret_aug_B = exact_policy_evaluation(solver, pi_aug_B)
                
                gap_iso_A = calc_exact_l_inf_distance(ret_iso_A, pareto_points, ranges)
                gap_iso_B = calc_exact_l_inf_distance(ret_iso_B, pareto_points, ranges)
                gap_iso = (gap_iso_A + gap_iso_B) / 2.0
                
                gap_aug_A = calc_exact_l_inf_distance(ret_aug_A, pareto_points, ranges)
                gap_aug_B = calc_exact_l_inf_distance(ret_aug_B, pareto_points, ranges)
                gap_aug = (gap_aug_A + gap_aug_B) / 2.0
                
                gaps["iso"].append(gap_iso)
                gaps["aug"].append(gap_aug)
                
            results["iso"].append((np.mean(gaps["iso"]), np.std(gaps["iso"])))
            results["aug"].append((np.mean(gaps["aug"]), np.std(gaps["aug"])))

        ax = axes[ax_idx]
        
        iso_means = np.array([m for m, s in results["iso"]])
        iso_stds  = np.array([s for m, s in results["iso"]])/np.sqrt(num_trials)
        aug_means = np.array([m for m, s in results["aug"]])
        aug_stds  = np.array([s for m, s in results["aug"]])/np.sqrt(num_trials)
        
        ax.plot(trajectory_steps, iso_means, 'o-', color='orange', linewidth=2.5, label='Isolated BC (Independent Experts)')
        ax.fill_between(trajectory_steps, np.maximum(0, iso_means - iso_stds), iso_means + iso_stds, color='orange', alpha=0.15, edgecolor='none')
        
        ax.plot(trajectory_steps, aug_means, 's-', color='green', linewidth=2.5, label='MA-BC (Ours)')
        ax.fill_between(trajectory_steps, np.maximum(0, aug_means - aug_stds), aug_means + aug_stds, color='green', alpha=0.15, edgecolor='none')
        
        if alpha == 0.0:
            title = "Agent always spawns at (0,0)"
        elif alpha == 1.0:
            title = "Agent spawns uniformly at random"
        else:
            title = f"{int(alpha*100)}% Chance of a random spawn"
            
        ax.set_title(title)
        ax.set_xlabel(r"Dataset Size per Expert")
        ax.set_xticks(trajectory_steps)
        ax.grid(True, linestyle=":", alpha=0.6)
        
        if ax_idx == 0:
            ax.set_ylabel(r"Normalized $L_\infty$ Distance to PF")
            ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        if ax_idx == 1:
            ax.legend(loc="upper right")
    
    fig.subplots_adjust(top=0.90, bottom=0.25, left=0.12, right=0.94)
    plt.savefig("dst_alpha_sweep.pdf", dpi=300, transparent=True)
    plt.show()

plot_shared_data_value_across_distributions(alphas=[0.2, 1.0], 
                                            trajectory_steps=[10, 20, 30, 40, 50, 60, 70, 80, 90, 100], 
                                            num_trials=100)

In [ ]:
def plot_alpha_sweep_fixed_N(alphas=np.linspace(0.0, 1.0, 11),
                             fixed_n_traj=40, 
                             num_trials=50):
    print("="*65)
    print(f" EVALUATING ALPHA SWEEP AT FIXED N = {fixed_n_traj} ")
    print("="*65 + "\n")
    
    iso_means, iso_sems = [], []
    aug_means, aug_sems = [], []

    for alpha in alphas:
        print(f"--- Processing Alpha = {alpha:.2f} ---")
        
        solver = DSTValueIteration(gamma=GAMMA, init_state_dist=alpha)
        
        ccs_results = solver.find_ccs_ols()
        pareto_points = np.array([r for r, p in ccs_results])
        pareto_policies = [p for r, p in ccs_results]
        
        ideal_point = np.max(pareto_points, axis=0)
        nadir_point = np.min(pareto_points, axis=0)
        ranges = np.clip(ideal_point - nadir_point, 1e-9, None)
        
        idx_A = 0
        idx_B = len(pareto_points) - 1
        pol_A_true = pareto_policies[idx_A]
        pol_B_true = pareto_policies[idx_B]

        gaps_iso, gaps_aug = [], []
        
        for _ in range(num_trials):
            dA, dB, sA, sB = generate_augmented_datasets(solver, pol_A_true, pol_B_true, fixed_n_traj)
            pi_iso_A = build_stochastic_policy(solver, dA)
            pi_iso_B = build_stochastic_policy(solver, dB)
            pi_aug_A = build_stochastic_policy(solver, dA + sB)
            pi_aug_B = build_stochastic_policy(solver, dB + sA)
            ret_iso_A = exact_policy_evaluation(solver, pi_iso_A)
            ret_iso_B = exact_policy_evaluation(solver, pi_iso_B)
            ret_aug_A = exact_policy_evaluation(solver, pi_aug_A)
            ret_aug_B = exact_policy_evaluation(solver, pi_aug_B)
            gap_iso_A = calc_exact_l_inf_distance(ret_iso_A, pareto_points, ranges)
            gap_iso_B = calc_exact_l_inf_distance(ret_iso_B, pareto_points, ranges)
            gaps_iso.append((gap_iso_A + gap_iso_B) / 2.0)
            
            gap_aug_A = calc_exact_l_inf_distance(ret_aug_A, pareto_points, ranges)
            gap_aug_B = calc_exact_l_inf_distance(ret_aug_B, pareto_points, ranges)
            gaps_aug.append((gap_aug_A + gap_aug_B) / 2.0)
            
        iso_means.append(np.mean(gaps_iso))
        iso_sems.append(np.std(gaps_iso) / np.sqrt(num_trials))
        aug_means.append(np.mean(gaps_aug))
        aug_sems.append(np.std(gaps_aug) / np.sqrt(num_trials))

    iso_means, iso_sems = np.array(iso_means), np.array(iso_sems)
    aug_means, aug_sems = np.array(aug_means), np.array(aug_sems)

    fig = plt.figure(figsize=(5, 5.2))
    ax = fig.gca()
    
    ax.plot(alphas, iso_means, 'o-', color='orange', linewidth=2.5, markersize=8, label='Isolated BC (Idependent Experts)')
    ax.fill_between(alphas, np.maximum(0, iso_means - iso_sems), iso_means + iso_sems, color='orange', alpha=0.15)
    
    ax.plot(alphas, aug_means, 's-', color='green', linewidth=2.5, markersize=8, label='MA-BC (Ours)')
    ax.fill_between(alphas, np.maximum(0, aug_means - aug_sems), aug_means + aug_sems, color='green', alpha=0.2)

    ax.set_title(fr"Deep Sea Treasure | $N_1=N_2={fixed_n_traj}$")
    ax.set_xlabel(r"Probability of Random Spawn")
    ax.set_ylabel(r"Normalized $L_\infty$ Distance to PF")
    
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    
    ax.set_xticks(np.linspace(0.0, 1.0, 6))
    
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper left", framealpha=0.95)
    ax.set_ylim(bottom=0)
    
    fig.subplots_adjust(top=0.90, bottom=0.25, left=0.24, right=0.98)

    plt.savefig("dst_alpha_sweep.pdf", dpi=300, transparent=True)
    plt.show()

alpha_values = np.linspace(0.0, 1.0, 11)
plot_alpha_sweep_fixed_N(alphas=alpha_values, fixed_n_traj=40, num_trials=100)

# Deep Dive

In [ ]:
def visualize_pareto_front(sorted_pareto_points):
    
    plt.figure(figsize=(12, 8))
    plt.plot(
        sorted_pareto_points[:, 0],
        sorted_pareto_points[:, 1],
        "b-",
        linewidth=2.5,
        zorder=6,
        label="Pareto Frontier",
    )
    plt.scatter(
        sorted_pareto_points[:, 0],
        sorted_pareto_points[:, 1],
        c="red",
        s=50,
        zorder=5,
        label="Pareto Front Vertices",
    )
    plt.title(f"Exact Pareto Front (gamma={GAMMA})")
    plt.xlabel("Discounted Treasure Value")
    plt.ylabel("Discounted Time Penalty")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper right")
    plt.show()
    print(f"Total Pareto vertices found: {len(sorted_pareto_points)}")

In [ ]:
visualize_pareto_front(sorted_pareto_points=sorted_pareto_points)

In [ ]:
def global_policy_audit(solver, all_results):
    print("="*65)
    print(" GLOBAL PARETO POLICY AUDIT & EXACT EDGE VERIFICATION ")
    print("="*65 + "\n")
    
    grouped_policies = defaultdict(list)
    for ret, pol in all_results:
        grouped_policies[tuple(np.round(ret, 5))].append((ret, pol))
        
    sorted_keys = sorted(grouped_policies.keys(), key=lambda x: (x[0], x[1]))
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    pareto_x = [k[0] for k in sorted_keys]
    pareto_y = [k[1] for k in sorted_keys]
    
    ax.plot(pareto_x, pareto_y, 'g-', linewidth=2, alpha=0.5, label="Optimal Convex Boundary (OLS Edges)")
    ax.scatter(pareto_x, pareto_y, c="green", s=120, edgecolor='black', zorder=5, label="Anchor Vertices")

    print("--- PART 2: ADJACENT VERTEX BRIDGES & EXACT SEQUENCE PROOF ---")
    
    plotted_stepping_stones = False
    plotted_alias_edge = False
    plotted_alias_marker = False
    
    for i in range(len(sorted_keys) - 1):
        key_A = sorted_keys[i]
        key_B = sorted_keys[i+1]
        
        ret_A, pol_A = grouped_policies[key_A][0]
        ret_B, pol_B = grouped_policies[key_B][0]
        
        diff_states = [s for s in pol_A if pol_A[s] != pol_B[s]]
        min_diff = len(diff_states)
        
        print(f"Bridge {i} -> {i+1} | {tuple(np.round(ret_A, 2))} to {tuple(np.round(ret_B, 2))}")
        
        if min_diff == 1:
            print(f"  -> Perfect Distance-1 Bridge. Flip State {diff_states[0]}: Action {pol_A[diff_states[0]]} -> {pol_B[diff_states[0]]}")
        else:
            print(f"  -> OLS policies differ by {min_diff} actions. Verifying the sequence...")
            
            wx = ret_B[1] - ret_A[1]
            wy = ret_A[0] - ret_B[0]
            w = np.abs(np.array([wx, wy]))
            w = w / np.sum(w)
            edge_val = np.dot(w, np.array(ret_A))
            
            curr_pol = pol_A.copy()
            
            aliases_A_count = 0
            aliases_B_count = 0
            stone_x, stone_y = [], []
            
            for step, s in enumerate(diff_states):
                curr_pol[s] = pol_B[s]
                inter_ret = solver.get_average_return(curr_pol)
                inter_val = np.dot(w, np.array(inter_ret))
                
                if np.allclose(inter_ret, ret_A, atol=1e-5):
                    status = "[ALIAS OF VERTEX A]"
                    aliases_A_count += 1
                elif np.allclose(inter_ret, ret_B, atol=1e-5):
                    status = "[ALIAS OF VERTEX B - JUMP CROSSED!]"
                    aliases_B_count += 1
                elif np.isclose(inter_val, edge_val, atol=1e-5):
                    status = "[ON FLAT EDGE - STEPPING STONE]"
                    stone_x.append(inter_ret[0])
                    stone_y.append(inter_ret[1])
                else:
                    status = "[OFF EDGE - DENT]"
                
                print(f"       Step {step+1} (Flip {s}): Return = {tuple(np.round(inter_ret, 2))} {status}")
            
            ghosts_A = aliases_A_count
            ghosts_B = max(0, aliases_B_count - 1)
            
            if stone_x:
                label = "Flat-Edge Stepping Stones" if not plotted_stepping_stones else ""
                ax.scatter(stone_x, stone_y, color="orange", marker="D", s=70, edgecolor="black", zorder=6, label=label)
                plotted_stepping_stones = True
            
            if ghosts_A > 0 or ghosts_B > 0:
                ax.annotate("", xy=(ret_B[0], ret_B[1]), xytext=(ret_A[0], ret_A[1]),
                            arrowprops=dict(arrowstyle="->", color="purple", lw=2.5, linestyle="--", alpha=0.9))
                
                if not plotted_alias_edge:
                    edge_label = "Alias Jump Edge (Instant Return Shift)"
                    ax.plot([], [], color="purple", lw=2.5, linestyle="--", label=edge_label)
                    plotted_alias_edge = True
                
                if ghosts_A > 0:
                    ax.scatter([ret_A[0]], [ret_A[1]], facecolors='none', edgecolors='purple', 
                            marker='o', s=300, linewidths=2, zorder=7)
                    ax.annotate(f"+{ghosts_A} Aliases", xy=(ret_A[0], ret_A[1]), xytext=(12, 12),
                                textcoords="offset points", fontsize=10, color="purple", fontweight="bold")
                    
                if ghosts_B > 0:
                    marker_label = "Aliased Policies (Ghost Flips)" if not plotted_alias_marker else ""
                    ax.scatter([ret_B[0]], [ret_B[1]], facecolors='none', edgecolors='purple', 
                            marker='o', s=300, linewidths=2, zorder=7, label=marker_label)
                    plotted_alias_marker = True
                    
                    ax.annotate(f"+{ghosts_B} Aliases", xy=(ret_B[0], ret_B[1]), xytext=(12, 12),
                                textcoords="offset points", fontsize=10, color="purple", fontweight="bold")

    print("\n" + "="*65)
    
    ax.set_title("1-Action Flips Between Pareto Vertices", fontsize=16)
    ax.set_xlabel("Discounted Treasure Value", fontsize=12)
    ax.set_ylabel("Discounted Time Penalty", fontsize=12)
    ax.grid(True, linestyle=":", alpha=0.6)
    
    ax.legend(loc="upper right", fontsize=11, framealpha=0.9)
    plt.tight_layout()
    plt.show()

In [ ]:
global_policy_audit(solver=full_solver, all_results=ccs_results)

In [ ]:
def export_pareto_front_audit(solver, ccs_results, filename="figs/dst_pareto_audit.pdf"):
    print("="*65)
    print(" GLOBAL PARETO POLICY AUDIT & EXACT EDGE VERIFICATION ")
    print("="*65 + "\n")
    
    grouped_policies = defaultdict(list)
    for ret, pol in ccs_results:
        grouped_policies[tuple(np.round(ret, 5))].append((ret, pol))
        
    sorted_keys = sorted(grouped_policies.keys(), key=lambda x: (x[0], x[1]))
    
    fig, ax = plt.subplots(figsize=(10, 5.2))
    
    pareto_x = [k[0] for k in sorted_keys]
    pareto_y = [k[1] for k in sorted_keys]
    
    ax.plot(pareto_x, pareto_y, 'g-', linewidth=3.5, zorder=4, label="Pareto Frontier")
    ax.scatter(pareto_x, pareto_y, c="green", s=150, edgecolor='black', zorder=5, label="Pareto Vertices (Deterministic Policies)")

    print("--- ADJACENT VERTEX BRIDGES & EXACT SEQUENCE PROOF ---")
    
    plotted_stepping_stones = False
    plotted_alias_edge = False
    plotted_alias_marker = False
    
    for i in range(len(sorted_keys) - 1):
        key_A, key_B = sorted_keys[i], sorted_keys[i+1]
        ret_A, pol_A = grouped_policies[key_A][0]
        ret_B, pol_B = grouped_policies[key_B][0]
        
        diff_states = [s for s in pol_A if pol_A[s] != pol_B[s]]
        min_diff = len(diff_states)
        
        print(f"Bridge {i} -> {i+1} | {tuple(np.round(ret_A, 2))} to {tuple(np.round(ret_B, 2))}")
        
        if min_diff == 1:
            print(f"  -> Perfect Distance-1 Bridge. Flip State {diff_states[0]}: Action {pol_A[diff_states[0]]} -> {pol_B[diff_states[0]]}")
        else:
            print(f"  -> OLS policies differ by {min_diff} actions. Verifying sequence...")
            
            wx = ret_B[1] - ret_A[1]
            wy = ret_A[0] - ret_B[0]
            w = np.abs(np.array([wx, wy]))
            w = w / np.sum(w)
            edge_val = np.dot(w, np.array(ret_A))
            
            curr_pol = pol_A.copy()
            aliases_A_count, aliases_B_count = 0, 0
            stone_x, stone_y = [], []
            
            for step, s in enumerate(diff_states):
                curr_pol[s] = pol_B[s]
                
                stoc_pol = {st: {ac: 1.0} for st, ac in curr_pol.items()}
                inter_ret = exact_policy_evaluation(solver, stoc_pol)
                inter_val = np.dot(w, np.array(inter_ret))
                
                if np.allclose(inter_ret, ret_A, atol=1e-4):
                    status = "[ALIAS OF VERTEX A]"
                    aliases_A_count += 1
                elif np.allclose(inter_ret, ret_B, atol=1e-4):
                    status = "[ALIAS OF VERTEX B - JUMP CROSSED!]"
                    aliases_B_count += 1
                elif np.isclose(inter_val, edge_val, atol=1e-4):
                    status = "[ON FLAT EDGE - STEPPING STONE]"
                    stone_x.append(inter_ret[0])
                    stone_y.append(inter_ret[1])
                else:
                    status = "[OFF EDGE - DENT]"
                
                print(f"       Step {step+1} (Flip {s}): Return = {tuple(np.round(inter_ret, 2))} {status}")
            
            ghosts_A = aliases_A_count
            ghosts_B = max(0, aliases_B_count - 1)
            
            if stone_x:
                label = "Flat-Edge Stepping Stones" if not plotted_stepping_stones else ""
                ax.scatter(stone_x, stone_y, color="orange", marker="D", s=100, edgecolor="black", zorder=6, label=label)
                plotted_stepping_stones = True
            
            if ghosts_A > 0 or ghosts_B > 0:
                ax.annotate("", xy=(ret_B[0], ret_B[1]), xytext=(ret_A[0], ret_A[1]),
                            arrowprops=dict(arrowstyle="->", color="purple", lw=3.0, linestyle="--", alpha=0.9))
                
                if ghosts_A > 0:
                    ax.scatter([ret_A[0]], [ret_A[1]], facecolors='none', edgecolors='purple', 
                               marker='o', s=450, linewidths=2.5, zorder=7)
                    ax.annotate(f"+{ghosts_A} Aliases", xy=(ret_A[0], ret_A[1]), xytext=(15, 15),
                                textcoords="offset points", fontsize=12, color="purple", fontweight="bold")
                    
                if ghosts_B > 0:
                    marker_label = "Aliased Policies (Same Returns)" if not plotted_alias_marker else ""
                    ax.scatter([ret_B[0]], [ret_B[1]], facecolors='none', edgecolors='purple', 
                               marker='o', s=450, linewidths=2.5, zorder=7, label=marker_label)
                    plotted_alias_marker = True
                    ax.annotate(f"+{ghosts_B} Aliases", xy=(ret_B[0], ret_B[1]), xytext=(15, 15),
                                textcoords="offset points", fontsize=12, color="purple", fontweight="bold")

    print("\n" + "="*65)
    
    ax.set_title("Deep Sea Treasure | Exact Pareto Front", pad=15)
    ax.set_xlabel("Discounted Treasure Value")
    ax.set_ylabel("Discounted Time Penalty")
    ax.grid(True, linestyle=":", alpha=0.6)
    
    ax.legend(loc="upper right", framealpha=0.95)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, transparent=True)
    plt.show()

export_pareto_front_audit(full_solver, ccs_results, filename="figs/dst_pareto_audit.pdf")

In [ ]:
def export_pareto_front_audit2(solver, ccs_results, pi_mixed, idx_A, idx_B, 
                              dense_front=None, front_meta=None, nadir=None, ranges=None, 
                              filename="figs/dst_pareto_audit.pdf"):
    print("="*65)
    print(" PARETO FRONT & MIXED POLICY VERTICAL GAP VERIFICATION ")
    print("="*65 + "\n")
    
    grouped_policies = defaultdict(list)
    for ret, pol in ccs_results:
        grouped_policies[tuple(np.round(ret, 5))].append((ret, pol))
        
    sorted_keys = sorted(grouped_policies.keys(), key=lambda x: (x[0], x[1]))
    pareto_points_sorted = np.array(sorted_keys)
    
    pareto_points_raw = np.array([r for r, p in ccs_results])
    
    fig, ax = plt.subplots(figsize=(10, 5.2))
    
    pareto_x = pareto_points_sorted[:, 0]
    pareto_y = pareto_points_sorted[:, 1]
    
    ax.plot(pareto_x, pareto_y, 'g-', linewidth=3.5, zorder=4, label="Pareto Frontier")
    ax.scatter(pareto_x, pareto_y, c="green", s=150, edgecolor='black', zorder=5, label="Pareto Vertices (Deterministic Policies)")

    ret_mixed = exact_policy_evaluation(solver, pi_mixed)
    x_mix, y_mix = ret_mixed[0], ret_mixed[1]
    
    y_proj = y_mix
    for i in range(len(pareto_points_sorted) - 1):
        x1, y1 = pareto_points_sorted[i]
        x2, y2 = pareto_points_sorted[i+1]
        
        if min(x1, x2) <= x_mix <= max(x1, x2):
            if x1 == x2:
                y_proj = max(y1, y2)
            else:
                y_proj = y1 + (x_mix - x1) * (y2 - y1) / (x2 - x1)
            break
    
    ax.scatter(*pareto_points_raw[idx_A], color='blue', marker='*', s=600, edgecolor='black', zorder=9, label="Time Expert")
    ax.scatter(*pareto_points_raw[idx_B], color='red', marker='*', s=600, edgecolor='black', zorder=9, label="Treasure Expert")
    
    ax.scatter(x_mix, y_mix, color='orange', marker='X', s=300, edgecolor='black', linewidth=1.5, zorder=10, label="Naive BC Expected Return")
    
    ax.plot([x_mix, x_mix], [y_mix, y_proj], color='black', linestyle='--', linewidth=2.5, zorder=8, label="Distance to PF")
    ax.scatter(x_mix, y_proj, color='white', marker='o', s=100, edgecolor='black', linewidth=2, zorder=8)

    print(f"Naice BC Return: ({x_mix:.2f}, {y_mix:.2f})")
    print(f"Distance to PF: ({x_mix:.2f}, {y_proj:.2f})")
    print(f"Absolute Gap in Time Penalty: {abs(y_proj - y_mix):.2f}\n")

    ax.set_title("Deep Sea Treasure | Expert Selection & Naive BC's Failure", pad=15)
    ax.set_xlabel("Discounted Treasure Value")
    ax.set_ylabel("Discounted Time Penalty")
    ax.grid(True, linestyle=":", alpha=0.6)
    
    ax.legend(loc="upper right", framealpha=0.95, fontsize=12, ncol=1)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, transparent=True)
    plt.show()

export_pareto_front_audit2(full_solver, ccs_results, pi_mixed, expert_1_idx, expert_2_idx,
                          dense_front, front_meta, nadir, ranges, filename="figs/dst_bc_failure.pdf")

In [ ]:
def export_mixed_vs_continuous_pareto(solver, pi_mixed, pareto_points, pareto_policies, 
                                      dense_front, front_meta, nadir, ranges, 
                                      filename="figs/dst_mixed_vs_optimal.pdf"):
    print("="*65)
    print(" ANALYZING MIXED MLE VS. CLOSEST CONTINUOUS PARETO EDGE ")
    print("="*65 + "\n")
    
    ret_mixed = exact_policy_evaluation(solver, pi_mixed)
    
    norm_mixed = (ret_mixed - nadir) / ranges
    distances = np.max(np.abs(dense_front - norm_mixed), axis=1)
    closest_idx = np.argmin(distances)
    
    v1_idx, v2_idx, best_alpha = front_meta[closest_idx]
    ret_closest = (1 - best_alpha) * pareto_points[v1_idx] + best_alpha * pareto_points[v2_idx]
    
    pi_closest = {}
    pol_1, pol_2 = pareto_policies[v1_idx], pareto_policies[v2_idx]
    for r in range(solver.rows):
        for c in range(solver.cols):
            s = (r, c)
            if s in pol_1 and s in pol_2:
                a1, a2 = pol_1[s], pol_2[s]
                if a1 == a2:
                    pi_closest[s] = {a1: 1.0}
                else:
                    pi_closest[s] = {a1: 1.0 - best_alpha, a2: best_alpha}
                    
    print(f"Mixed Policy Exact Return: {tuple(np.round(ret_mixed, 2))}")
    print(f"Projected to Edge {v1_idx} -> {v2_idx} (Alpha = {best_alpha:.2f})")
    print(f"Optimal Edge Return      : {tuple(np.round(ret_closest, 2))}\n")

    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
    fig.patch.set_facecolor('#f8f9fa')
    
    title_mix = "Naive BC Policy"
    title_opt = "Closest Pareto Optimal Policy"
    
    draw_policy_on_grid(axes[0], solver, pi_mixed, title=title_mix, highlight_conflicts=True, is_mixed_eval=True)
    draw_policy_on_grid(axes[1], solver, pi_closest, title=title_opt, highlight_conflicts=True, is_mixed_eval=False)
    
    custom_lines = [
        patches.Patch(color='white', label='Deterministic Action'),
        patches.Patch(color='#ffe6e6', label='Bad Mix (Mode Collapse)'),
        patches.Patch(color='#e6ffe6', label='Optimal Mix (Pareto Edge)'),
    ]
    
    fig.legend(handles=custom_lines, loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.02))
    
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    
    plt.savefig(filename, dpi=300, transparent=True)
    plt.show()

export_mixed_vs_continuous_pareto(full_solver, pi_mixed, sorted_pareto_points, sorted_pareto_policies, 
                                  dense_front, front_meta, nadir, ranges, 
                                  filename="figs/dst_mixed_vs_optimal.pdf")

In [ ]:

def export_isolated_vs_augmented_grid(solver, pi_iso, pi_aug, ret_iso, ret_aug, 
                                      filename="figs/dst_iso_vs_aug.pdf"):
    print("="*65)
    print(" VISUALIZING ISOLATED VS. AUGMENTED BEHAVIORAL CLONING ")
    print("="*65 + "\n")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
    fig.patch.set_facecolor('#f8f9fa')
    
    title_iso = fr"Isolated BC (Time Expert Data Only) | $N_1=N_2={50}$"
    draw_policy_on_grid(axes[0], solver, pi_iso, title=title_iso)
    
    title_aug = fr"MA-BC (Time Expert Data + Shared Treasure Expert Data) | $N_1=N_2={50}$"
    draw_policy_on_grid(axes[1], solver, pi_aug, title=title_aug)
    
    custom_lines = [
        patches.Patch(color='white', label='Seen State'),
        patches.Patch(color='#ffcccc', label='Unseen State (Uniformly Random Policy)'),
        patches.Patch(color='#ffd700', label='Treasure Target')
    ]
    
    fig.legend(handles=custom_lines, loc='lower center', ncol=3, fontsize=16, bbox_to_anchor=(0.5, 0.02))
    
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.savefig(filename, dpi=300, transparent=True)
    plt.show()
    print(f"Saved: {filename}\n")

n_traj_visual = 50
print(f"Generating sparse datasets ({n_traj_visual} trajectories) to highlight the coverage gap...")
dA_vis, dB_vis, sA_vis, sB_vis = generate_augmented_datasets(full_solver, pol_A_true, pol_B_true, n_traj_visual)

pi_iso_A_vis = build_stochastic_policy(full_solver, dA_vis)
pi_aug_A_vis = build_stochastic_policy(full_solver, dA_vis + sB_vis)

ret_iso_A_vis = exact_policy_evaluation(full_solver, pi_iso_A_vis)
ret_aug_A_vis = exact_policy_evaluation(full_solver, pi_aug_A_vis)

export_isolated_vs_augmented_grid(full_solver, pi_iso_A_vis, pi_aug_A_vis, 
                                  ret_iso_A_vis, ret_aug_A_vis)